# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema hosted at:
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load dataset metadata and create mlcroissant Dataset object
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset title:", metadata.name)
print("Description:", metadata.description)
print("Identifier:", metadata.identifier)
print("License:", metadata.license)
print("Number of authors:", len(metadata.author))
print("Date Published:", metadata.datePublished)

## 2. Data Overview
Review available record sets, their `@id`s, and available fields and columns.

In [ ]:
# List all available record sets and their fields by @id
record_sets_info = dataset.record_sets()

print("Available Record Sets:")
for rs in record_sets_info:
    print(f"- Record Set @id: {rs['@id']}, Name: {rs.get('name','Unknown')}")
    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print("  Fields:")
    for fld in fields:
        if isinstance(fld, dict):
            print(f"    - Field @id: {fld.get('@id', str(fld))}, Name: {fld.get('name','N/A')}")
        else:
            print(f"    - Field @id: {str(fld)}")
    print("---")
# For demonstration, pick the first record set @id
if record_sets_info:
    example_record_set_id = record_sets_info[0]['@id']
    print("\nSample of records from the first record set:")
    for i, rec in enumerate(dataset.records(record_set=example_record_set_id)):
        print(rec)
        if i > 2:  # Print only the first 3 records
            break

## 3. Data Extraction
Load data from all available record sets into pandas DataFrames for analysis.
We reference each record set and field by their `@id` as per Croissant schema.

In [ ]:
# Create a list of all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets_info]

dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records for Record Set @id: {rs_id}")

# Show columns of the first record set
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print("Columns for first record set:", dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, grouping.
Here, we select a numeric field, filter records, normalize values, and group by another field.
- All fields referenced by `@id` (column name).

In [ ]:
# Choose record set and numeric and group field for analysis.
rs_id = first_rs_id
df = dataframes[rs_id]
# Print candidate numeric fields
print("Numeric fields available:")
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        print(f"- {col}")
# For demonstration, pick a column with 'Age' in name if present (from 'personalSensitiveInformation')
numeric_field_id = None
for col in df.columns:
    if 'age' in col.lower():
        numeric_field_id = col
        break
if numeric_field_id is None:
    # fallback to first numeric column
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
    else:
        print("No numeric fields found!")
        numeric_field_id = df.columns[0]

print(f"Selected numeric field @id: {numeric_field_id}")

# Filtering: threshold for age or numeric value
threshold = 50
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping: candidate group field (sex, anatomical location, etc.)
group_field_id = None
for col in df.columns:
    if 'sex' in col.lower() or 'location' in col.lower():
        group_field_id = col
        break

if group_field_id:
    print(f"Grouping by field @id: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    display(grouped_df.head())
else:
    print("No grouping field found.")

## 5. Visualization
Visualize data distributions and relationships between fields.
- Use only field names corresponding to `@id` in the Croissant schema.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id], bins=10, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# If group_field_id exists, boxplot
if group_field_id:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
This notebook demonstrated loading and processing the FAIR^2 dataset using the Croissant schema.

- Data was loaded and referenced by Croissant `@id` throughout.
- Numeric fields were filtered, normalized, and grouped for exploratory analysis.
- Visualizations provided insight into distributions and relationships in the data.

For further exploration, consult the dataset schema for detailed descriptions of each field and record set `@id`, and extend analyses accordingly.